In [1]:
# import re

# def extract_questions(text):
#     lines = text.split("\n")
#     questions = []
#     for line in lines:
#         line = line.strip()
#         if re.match(r"^(\d+\.|-|\*)\s+", line):
#             questions.append(re.sub(r"^(\d+\.|-|\*)\s+", "", line))
#         elif line.endswith("?"):
#             questions.append(line)
#     return questions
  

# import ollama

# client = ollama.Client()
# MODEL_NAME = "llama3.2:1b"


# file_content = """// db.js
# import mysql from "mysql2/promise";

# export const pool = mysql.createPool({
#   host: "localhost",
#   user: "root",
#   password: "YOUR_PASSWORD",
#   database: "hello",
#   connectionLimit: 10,
# });
# """

# prompt = (
#     "Read the following text and generate a list of clear, helpful questions "
#     "that someone might ask to better understand it.\n\n"
#     f"--- TEXT START ---\n{file_content}\n--- TEXT END ---\n\n"
#     "Questions:"
# )

# response = client.generate(model=MODEL_NAME, prompt=prompt)
# raw_output = response["response"]
# print(raw_output)
# questions = extract_questions(raw_output)
# questions

## Section: Creation of input[Foldername].json
### Run below only once because it creates teh input.json

In [1]:
import os
import json
import ollama

client = ollama.Client()
MODEL_NAME = "llama3.2:1b"

# Directory containing your code files
directory_path = "./More-code-files"  # <-- change to your folder

# Extract the directory name from the path
dir_name = os.path.basename(os.path.normpath(directory_path))  # Gets "BI-backend"

# Create the new output filename
output_file = f"./input__json files/input{dir_name}.json"

print(f"output_file = \"{output_file}\"")
# Output: output_file = "inputBI-backend.json"

input_data = []

for filename in os.listdir(directory_path):
    file_path = os.path.join(directory_path, filename)
    
    if os.path.isfile(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            file_content = f.read()
        
        prompt = (
            "Read the following text and generate a list of clear, concise, helpful questions "
            "that someone might ask to better understand it.\n\n"
            f"--- TEXT START ({filename}) ---\n{file_content}\n--- TEXT END ---\n\n"
            "Return only a JSON array of questions (strings)."
        )
        
        response = client.generate(model=MODEL_NAME, prompt=prompt)
        raw_output = response["response"].strip()
        
        # Try to parse JSON output safely
        try:
            questions = json.loads(raw_output)
            if not isinstance(questions, list):
                raise ValueError
        except Exception:
            # fallback: split by newlines if JSON fails
            questions = [q.strip("- ").strip() for q in raw_output.split("\n") if q.strip()]
        
        # Append to input data
        input_data.append({
            "file_name": filename,
            "code": file_content,
            "questions": questions
        })
        
        print(f"Generated {len(questions)} questions for {filename}")

# Save all data to input.json
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(input_data, f, indent=2)

print(f"\nAll files processed. Input JSON saved to {output_file}")


output_file = "./input__json files/inputMore-code-files.json"
Generated 7 questions for 2 Using 'cart_timestamp' to display items in certain times so that people add to cart .ipynb
Generated 94 questions for 3 Using 'orders' to recommend items.ipynb
Generated 5 questions for App.jsx
Generated 12 questions for Cards.tsx
Generated 8 questions for Charts.jsx
Generated 14 questions for famvitamincintake.py
Generated 20 questions for gameCreatorTool.html
Generated 29 questions for habittrainer.ipynb
Generated 9 questions for indexing.py
Generated 13 questions for machineAmachineB.py
Generated 5 questions for main.jsx
Generated 10 questions for preprocess_documents.py
Generated 11 questions for query_processing.py
Generated 10 questions for ranking.py
Generated 9 questions for searchEngine.py
Generated 12 questions for server.js
Generated 14 questions for simpleSearchEngine.py
Generated 32 questions for simplex.py
Generated 17 questions for tablechair.py
Generated 10 questions for Tables.tsx

In [4]:
# Now the input.json is created

OpenAI evaluation

In [5]:
# import json
# import re
# import hashlib
# from typing import Dict, List
# from openai import OpenAI

# # =========================
# # CONFIG
# # =========================
# MODEL = "gpt-4.1-mini"
# LLM_WEIGHT = 0.6
# RULE_WEIGHT = 0.4
# MIN_LENGTH = 5

# client = OpenAI()

# # =========================
# # RULE-BASED SCORING
# # =========================
# def rule_score(code: str, question: str) -> Dict[str, int]:
#     score = {"relevance": 1, "specificity": 1, "depth": 1, "clarity": 1}

#     identifiers = set(re.findall(r"\b[a-zA-Z_][a-zA-Z0-9_]*\b", code))
#     literals = re.findall(r"['\"](.*?)['\"]|\b\d+\b", code)

#     if any(id.lower() in question.lower() for id in identifiers):
#         score["relevance"] += 2
#         score["specificity"] += 1

#     if any(str(lit) in question for lit in literals):
#         score["specificity"] += 2

#     if re.search(r"\b(why|difference|trade[- ]off|advantage|impact)\b", question, re.I):
#         score["depth"] += 3

#     if len(question.split()) >= MIN_LENGTH and question.endswith("?"):
#         score["clarity"] += 3

#     return {k: min(v, 5) for k, v in score.items()}

# # =========================
# # LLM-BASED SCORING
# # =========================
# def llm_score(code: str, question: str) -> Dict[str, int]:
#     prompt = f"""
# You are evaluating the quality of a question about the following code.

# CODE:
# {code}

# QUESTION:
# {question}

# Score from 1–5 on:
# - relevance
# - specificity
# - depth
# - clarity

# Return JSON only.
# """

#     response = client.responses.create(
#         model=MODEL,
#         input=prompt,
#         temperature=0
#     )

#     text = response.output_text
#     return json.loads(text)

# # =========================
# # COMBINED SCORING
# # =========================
# def combine_scores(rule: Dict, llm: Dict) -> Dict:
#     combined = {}
#     total = 0

#     for key in ["relevance", "specificity", "depth", "clarity"]:
#         combined[key] = round(
#             RULE_WEIGHT * rule[key] + LLM_WEIGHT * llm[key], 2
#         )
#         total += combined[key]

#     combined["total"] = round(total, 2)
#     return combined

# # =========================
# # MAIN PIPELINE
# # =========================
# def score_file(data: Dict) -> Dict:
#     results = []

#     for q in data["questions"]:
#         rule = rule_score(data["code"], q)
#         llm = llm_score(data["code"], q)
#         final = combine_scores(rule, llm)

#         results.append({
#             "question": q,
#             "rule_score": rule,
#             "llm_score": llm,
#             "final_score": final
#         })

#     avg = round(sum(r["final_score"]["total"] for r in results) / len(results), 2)

#     return {
#         "file": data["file_name"],
#         "average_score": avg,
#         "questions": results
#     }

# # =========================
# # RUN
# # =========================
# if __name__ == "__main__":
#     with open("input.json") as f:
#         files = json.load(f)

#     reports = [score_file(file) for file in files]

#     with open("report.json", "w") as f:
#         json.dump(reports, f, indent=2)

#     print("Scoring complete → report.json")


Gemini version

In [ ]:
# import json
# import re
# from typing import Dict, List
# import google.generativeai as genai

# from dotenv import load_dotenv
# import os
# import google.generativeai as genai




# load_dotenv()  # loads variables from .env
# api_key = os.getenv("GENERATIVEAI_API_KEY")
# genai.configure(api_key=api_key)


c:\Users\HP\Documents\GitHub\25-26J-087\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\HP\AppData\Local\Temp\ipykernel_22700\3505649446.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


# Section: Available models

In [ ]:

# for m in genai.list_models():
#     if 'generateContent' in m.supported_generation_methods:
#         print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-exp-1206
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image-preview
models/gemini-2.5-flash-image
models/gemini-2.5-flash-preview-09-2025
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-prev

In [ ]:
# MODEL = "models/gemini-2.5-flash"  # Free-tier Gemini model
# LLM_WEIGHT = 0.6
# RULE_WEIGHT = 0.4
# MIN_LENGTH = 5
# # =========================
# # LLM-BASED SCORING (Gemini)
# # =========================
# def llm_score(code: str, question: str) -> Dict[str, int]:
#     prompt = f"""
# You are evaluating the quality of a question about the following code.

# CODE:
# {code}

# QUESTION:
# {question}

# Score from 1–5 on:
# - relevance
# - specificity
# - depth
# - clarity

# Return JSON only.
# """

#     # Create the model object
#     model = genai.GenerativeModel(MODEL)  

#     # Send a single content generation request
#     response = model.generate_content(prompt)

#     # Get the text string
#     text_output = response.text

#     try:
#         return json.loads(text_output)
#     except json.JSONDecodeError:
#         return {
#             "relevance": 3,
#             "specificity": 3,
#             "depth": 3,
#             "clarity": 3
#         }

simple usage of above

In [9]:
# # Example Python code snippet
# code_snippet = """
# def add(a, b):
#     return a + b
# """

# # Example question about the code
# question_text = "How can I modify this function to handle adding multiple numbers?"

# # Call the LLM scoring function
# scores = llm_score(code_snippet, question_text)

# print(scores)


In [ ]:
# # =========================
# # RULE-BASED SCORING
# # =========================
# def rule_score(code: str, question: str) -> Dict[str, int]:
#     score = {"relevance": 1, "specificity": 1, "depth": 1, "clarity": 1}

#     identifiers = set(re.findall(r"\b[a-zA-Z_][a-zA-Z0-9_]*\b", code))
#     literals = re.findall(r"['\"](.*?)['\"]|\b\d+\b", code)
    
#     if any(id.lower() in question.lower() for id in identifiers):
#         score["relevance"] += 2
#         score["specificity"] += 1

#     if any(str(lit) in question for lit in literals):
#         score["specificity"] += 2

#     if re.search(r"\b(why|difference|trade[- ]off|advantage|impact)\b", question, re.I):
#         score["depth"] += 3

#     if len(question.split()) >= MIN_LENGTH and question.endswith("?"):
#         score["clarity"] += 3

#     return {k: min(v, 5) for k, v in score.items()}


# def combine_scores(rule: Dict, llm: Dict) -> Dict:
#     combined = {}
#     total = 0

#     for key in ["relevance", "specificity", "depth", "clarity"]:
#         combined[key] = round(
#             RULE_WEIGHT * rule[key] + LLM_WEIGHT * llm[key], 2
#         )
#         total += combined[key]

#     combined["total"] = round(total, 2)
#     return combined

# def score_file(data: Dict) -> Dict:
#     results = []

#     for q in data["questions"]:
#         rule = rule_score(data["code"], q)
#         llm = llm_score(data["code"], q)
#         final = combine_scores(rule, llm)

#         results.append({
#             "question": q,
#             "rule_score": rule,
#             "llm_score": llm,
#             "final_score": final
#         })

#     avg = round(sum(r["final_score"]["total"] for r in results) / len(results), 2)

#     return {
#         "file": data["file_name"],
#         "average_score": avg,
#         "questions": results
#     }


### Section Creating reportGemini.json

In [ ]:
# if __name__ == "__main__":
#     with open("inputFinal.json") as f:
#         files = json.load(f)

#     reports = [score_file(file) for file in files]

#     with open("reportGemini.json", "w") as f:
#         json.dump(reports, f, indent=2)

#     print("Scoring complete → report.json")


Scoring complete → report.json


### Section: Creating manual_scores_Final.json

In [3]:
print("""
1. Relevance

“How directly it relates to the given code”

Is the question/comment actually about *thiscode?
Does it reference logic, functions, variables, or behavior that exists here?
❌ Not relevant: asking about general programming theory unrelated to the file
✅ Relevant: asking why a specific function is structured a certain way

2. Specificity

“How tightly it is grounded in this file (not generic)”

Does it point to concrete details in the file?
Does it mention exact functions, lines, or behaviors?
❌ Not specific: “Why is this written this way?”
✅ Specific: “Why does `processData()` mutate `state` instead of returning a new object?”

3. Depth

“Does it promote meaningful understanding or reasoning?”

Does it encourage thinking about *whysomething works, not just *whatit does?
Does it uncover tradeoffs, design decisions, or consequences?
❌ Shallow: “What does this function do?”
✅ Deep: “What are the tradeoffs of using recursion here instead of iteration?”

4. Clarity

“Is the question clear and unambiguous?”

Can the reader understand it immediately?
Is there only one reasonable interpretation?
❌ Unclear: “Is this right?”
✅ Clear: “Is this error handling sufficient if the API returns malformed JSON?”
""")


1. Relevance

“How directly it relates to the given code”

Is the question/comment actually about *thiscode?
Does it reference logic, functions, variables, or behavior that exists here?
❌ Not relevant: asking about general programming theory unrelated to the file
✅ Relevant: asking why a specific function is structured a certain way

2. Specificity

“How tightly it is grounded in this file (not generic)”

Does it point to concrete details in the file?
Does it mention exact functions, lines, or behaviors?
❌ Not specific: “Why is this written this way?”
✅ Specific: “Why does `processData()` mutate `state` instead of returning a new object?”

3. Depth

“Does it promote meaningful understanding or reasoning?”

Does it encourage thinking about *whysomething works, not just *whatit does?
Does it uncover tradeoffs, design decisions, or consequences?
❌ Shallow: “What does this function do?”
✅ Deep: “What are the tradeoffs of using recursion here instead of iteration?”

4. Clarity

“Is the que

In [ ]:
data = [{
    "file_name": "query_processing.py",
    "code": "import nltk\nimport string\nfrom nltk.corpus import wordnet\nfrom nltk.corpus import stopwords\nfrom nltk.stem import WordNetLemmatizer\n\n# NLTK data files download\nnltk.download('punkt')\nnltk.download('stopwords')\nnltk.download('wordnet')\n\ndef preprocess_query(query):\n    # Lowercase the query\n    query = query.lower()\n    # Remove punctuation (keep numbers)\n    punctuations = string.punctuation.replace('-', '')\n    query = query.translate(str.maketrans('', '', punctuations))\n    # Tokenize the query\n    tokens = nltk.word_tokenize(query)\n    # Remove stopwords\n    stop_words = set(stopwords.words('english'))\n    tokens = [token for token in tokens if token not in stop_words]\n    # Lemmatization\n    lemmatizer = WordNetLemmatizer()\n    tokens = [lemmatizer.lemmatize(token) for token in tokens]\n    # Join tokens back to string\n    preprocessed_query = ' '.join(tokens)\n    return preprocessed_query\n\ndef query_expansion(tokens):\n    expanded_tokens = tokens.copy()\n    for token in tokens:\n        synonyms = wordnet.synsets(token)\n        for syn in synonyms:\n            for lemma in syn.lemmas():\n                expanded_tokens.append(lemma.name())\n    # Remove duplicates\n    expanded_tokens = list(set(expanded_tokens))\n    return expanded_tokens\n\nif __name__ == '__main__':\n    query = input(\"Enter your search query: \")\n    preprocessed_query = preprocess_query(query)\n    tokens = preprocessed_query.split()\n    expanded_tokens = query_expansion(tokens)\n    expanded_query = ' '.join(expanded_tokens)\n    print(\"Expanded Query:\", expanded_query)",
    "questions": [
      "What is the purpose of the script and what tasks does it perform?",
      "How do I download the required NLTK data files if they're not already downloaded?",
      "Can you explain why you've used certain NLTK functions (e.g. `punkt`, `stopwords`, `wordnet`) in this script?",
      "How does the `preprocess_query` function handle punctuation and numbers in a query string?",
      "What is the difference between the original tokens, the preprocessed tokens, and the expanded tokens?",
      "Can you provide an example of how to use the `query_expansion` function to expand synonyms for a given token?",
      "Why are stopwords removed from the token list before lemmatization? Is there a specific reason or a predefined rule?",
      "How does the script handle case sensitivity when working with queries and tokens?",
      "Can you explain why duplicates in the expanded tokens set are being removed using `set(expanded_tokens)`?",
      "What is the purpose of the final `print` statement that prints the \"Expanded Query\" value?"
    ]
  }]


def manual_scores(question):
    print(f"\nQuestion: {question}")
    return {
        "Relevance": int(input(f"\nQuestion: {question}::::::::Relevance (1–5): ")),
        "Specificity": int(input(f"\nQuestion: {question}::::::::Specificity (1–5): ")),
        "Depth": int(input(f"\nQuestion: {question}::::::::Depth (1–5): ")),
        "clarity": int(input(f"\nQuestion: {question}::::::::Clarity (1–5): ")),
    }
    
  
all_scores = {}

global_file_name = ""
for file in data:
    file_name = file["file_name"]
    global_file_name = file_name
    all_scores[file_name] = []

    for question in file["questions"]:
        scores = manual_scores(question)
        all_scores[file_name].append({
            "question": question,
            "scores": scores
        })

import json

manual_scores_json = "./manual_scores__json files/"+ "manual_scores_"+ global_file_name +".json"
with open(manual_scores_json, "w") as f:
    json.dump(all_scores, f, indent=2)



Question: What is the purpose of the script and what tasks does it perform?

Question: How do I download the required NLTK data files if they're not already downloaded?

Question: Can you explain why you've used certain NLTK functions (e.g. `punkt`, `stopwords`, `wordnet`) in this script?

Question: How does the `preprocess_query` function handle punctuation and numbers in a query string?

Question: What is the difference between the original tokens, the preprocessed tokens, and the expanded tokens?

Question: Can you provide an example of how to use the `query_expansion` function to expand synonyms for a given token?

Question: Why are stopwords removed from the token list before lemmatization? Is there a specific reason or a predefined rule?

Question: How does the script handle case sensitivity when working with queries and tokens?

Question: Can you explain why duplicates in the expanded tokens set are being removed using `set(expanded_tokens)`?

Question: What is the purpose of t